# Cálculo de la capacidad y demanda

Usa un **horizonte** $H$ (días) y expresa **todo en “personas atendidas en $H$"**: convierte la capacidad de cada hospital $j$ con $C_j=\text{camas}_j\cdot\frac{H}{LOS}\cdot occ$ (cada cama atiende $\frac{H}{LOS}$ personas en $H$, ajustado por la ocupación objetivo $occ$), y la **demanda** de cada ciudad $i$ como $q_i=\text{población}_i\cdot\rho$ (fracción $\rho$ que requerirá esa atención en $H$). Antes de optimizar, verifica factibilidad global $\sum_i q_i \le \sum_j C_j$; si no se cumple, ajusta $\rho$, $H$, $LOS$ o reescala la demanda con $\alpha=\frac{\sum_j C_j}{\sum_i q_i}$ (i.e., $q_i\leftarrow \alpha,q_i$).


In [1]:
import pandas as pd

# Parámetros del horizonte
H   = 30       # días
LOS = 5        # días de estancia media por paciente
occ = 0.85     # ocupación objetivo (85%)
rho = 0.01     # fracción de la población que requerirá ingreso/atención en H

# Carga
ciudades   = pd.read_csv("../data/processed/andaluces_2_5k.csv")          # debe tener 'poblacion'
hospitales = pd.read_csv("../data/processed/Hospitales_Completo.csv")     # debe tener 'capacidad'

# Demanda: personas que requerirán la atención en H (mismo tipo que el recurso)
ciudades["q"] = ciudades["poblacion"] * rho

# Capacidad: personas que pueden ser atendidas en H con rotación por LOS y ocupación
hospitales["C"] = hospitales["capacidad"] * (H / LOS) * occ

# Chequeo de factibilidad global
Q = ciudades["q"].sum()
Ctot = hospitales["C"].sum()
print(f"Demanda total esperada (personas en {H} días): {Q:.1f}")
print(f"Capacidad total disponible (personas en {H} días): {Ctot:.1f}  ->  Q/C = {Q/Ctot:.2f}")

# (Opcional) Reescalar demanda si no cabe
if Q > Ctot:
    factor = Ctot / Q
    ciudades["q"] *= factor
    print(f"Demanda reescalada por factor {factor:.3f} para cumplir factibilidad global.")

# Guardar para tu GA
ciudades.to_csv("../data/processed/Ciudades_Con_Demanda.csv", index=False)
hospitales.to_csv("../data/processed/Hospitales_Con_Capacidad.csv", index=False)

Demanda total esperada (personas en 30 días): 81858.1
Capacidad total disponible (personas en 30 días): 117019.5  ->  Q/C = 0.70


# Cálculo matriz distancias

Antes, entrar en powershell como administrador y ejecutar 
docker run -t -i -p 5000:5000 -v "C:\Users\pedro\Desktop\master\Optimización Computacional\Trabajo\osrm-andalucia:/data" osrm/osrm-backend osrm-routed --algorithm ch /data/andalucia-latest.osrm


In [7]:
import pandas as pd
import numpy as np
import requests

ciudades = pd.read_csv('andaluces_2_5k.csv')
hospitales = pd.read_csv('Hospitales_Completo.csv')

city_coords = list(zip(ciudades['longitud'], ciudades['latitud']))
hosp_coords = list(zip(hospitales['longitud'], hospitales['latitud']))

def osrm_table_batched(city_coords, hosp_coords, osrm_url="http://localhost:5000",
                       batch_size=75, max_table_size=10000):
    """
    Construye matrices ciudad-hospital usando OSRM /table en bloques.
    Ajusta batch_size para respetar max_table_size = #sources * #destinations.
    """
    n_cities = len(city_coords)
    n_hosps = len(hosp_coords)

    # Asegurar que el batch no viola el límite de OSRM
    max_sources = max_table_size // n_hosps
    if batch_size > max_sources:
        batch_size = max_sources
        print(f"batch_size ajustado a {batch_size} para respetar max_table_size={max_table_size}")

    dist_km = np.zeros((n_cities, n_hosps))
    time_min = np.zeros((n_cities, n_hosps))

    for start in range(0, n_cities, batch_size):
        end = min(start + batch_size, n_cities)
        batch = city_coords[start:end]
        local_n = end - start

        all_coords = batch + hosp_coords
        coords_str = ";".join(f"{lon},{lat}" for lon, lat in all_coords)

        sources = ";".join(str(i) for i in range(local_n))
        destinations = ";".join(str(local_n + j) for j in range(n_hosps))

        url = (
            f"{osrm_url}/table/v1/driving/{coords_str}"
            f"?sources={sources}&destinations={destinations}&annotations=distance,duration"
        )

        r = requests.get(url)
        # Si algo falla, muestra el mensaje de OSRM para entenderlo
        if not r.ok:
            print("Error en batch", start, end)
            print("Status code:", r.status_code)
            try:
                print("Respuesta OSRM:", r.json())
            except Exception:
                print("Contenido:", r.text[:500])
            r.raise_for_status()

        data = r.json()

        dist_block = np.array(data["distances"]) / 1000.0   # m -> km
        time_block = np.array(data["durations"]) / 60.0     # s -> min

        dist_km[start:end, :] = dist_block
        time_min[start:end, :] = time_block

        print(f"Batch {start}:{end} OK")

    return dist_km, time_min

dist_matrix_km, time_matrix_min = osrm_table_batched(city_coords, hosp_coords)

print(dist_matrix_km.shape, time_matrix_min.shape)


Batch 0:75 OK
Batch 75:150 OK
Batch 150:225 OK
Batch 225:300 OK
Batch 300:375 OK
Batch 375:406 OK
(406, 133) (406, 133)


In [8]:
np.savetxt("dist_ciudad_hospital_km.csv", dist_matrix_km, delimiter=",")
np.savetxt("time_ciudad_hospital_min.csv", time_matrix_min, delimiter=",")


# Optimización